# 210 · Paper v3 — Final Analysis (GOOG + INTC, Jan 2026)

**Out-of-sample evaluation on January 2026 data.**
Fixes data leakage from v2 (training data included Jan 2023 test period).

Additions over NB 201:
- Null baseline (no-injection control)
- Hurst exponent of order flow
- Propagator function G(l)
- Spread dynamics during impact
- Cross-stock validation (Intel INTC)
- Parameter sensitivity (mb dominates beta)

In [ ]:
import numpy as np
import pandas as pd
import re, gc, math, json
from pathlib import Path
from collections import OrderedDict
from scipy import stats, optimize
from scipy.signal import savgol_filter
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# -- Publication figure style --
SINGLE_W = 520
FULL_W   = 1080
FIG_H    = 400
TEMPLATE = "plotly_white"
FONT     = dict(family="Times New Roman, serif", size=14)
SAVE_DIR = Path("pics_for_210_paper_v3")
SAVE_DIR.mkdir(exist_ok=True)

def save_fig(fig, name, w=FULL_W, h=FIG_H):
    fig.write_image(SAVE_DIR / f"{name}.png", width=w, height=h, scale=3)
    fig.write_image(SAVE_DIR / f"{name}.pdf", width=w, height=h)
    print(f"  Saved {name}")

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────
TICK_SIZE = 100
MAX_SAMPLES = 2048
MIDPRICE_MAX = 2_000_000
N_BOOTSTRAP = 1000
N_COND_MSGS = 500
GRID = 'c10x_v2'

# -- Model metadata --
MODEL_META = OrderedDict([
    ("Historic",  dict(color="#999999", dash="dot",    marker="x")),
    ("Heuristic", dict(color="#D4A017", dash="dashdot",marker="diamond")),
    ("CST",       dict(color="#E07B39", dash="dash",   marker="triangle-up")),
    ("CGAN",      dict(color="#8B4513", dash="longdash",marker="square")),
    ("LobS5",    dict(color="#1B9E77", dash="solid",  marker="circle")),
    ("S5-120M",   dict(color="#D95F02", dash="solid",  marker="triangle-down")),
    ("S5-4K",     dict(color="#7570B3", dash="solid",  marker="star")),
])

# -- Paths: GOOG Jan 2026 (update after Isambard runs) --
GOOG_PATHS = OrderedDict([
    ("Historic",  dict(buy="PATH/context_500_buy", sell="PATH/context_500_sell")),
    ("Heuristic", dict(buy="PATH/context_500_buy", sell="PATH/context_500_sell")),
    ("CST",       dict(buy="PATH/context_500_buy", sell="PATH/context_500_sell")),
    ("CGAN",      dict(buy="PATH/context_500_buy", sell="PATH/context_500_sell")),
    ("LobS5",    dict(buy="PATH/context_500_buy", sell="PATH/context_500_sell")),
    ("S5-120M",   dict(buy="PATH/context_500_buy", sell="PATH/context_500_sell")),
    ("S5-4K",     dict(buy="PATH/context_500_buy", sell="PATH/context_500_sell")),
])

# -- Paths: INTC Jan 2026 (update after Isambard runs) --
INTC_PATHS = OrderedDict([
    ("LobS5",    dict(buy="PATH/context_500_buy", sell="PATH/context_500_sell")),
    ("S5-120M",   dict(buy="PATH/context_500_buy", sell="PATH/context_500_sell")),
    ("S5-4K",     dict(buy="PATH/context_500_buy", sell="PATH/context_500_sell")),
])

# -- Paths: Null baseline --
NULL_PATHS = OrderedDict([
    ("LobS5",    "PATH/null_baseline/LobS5/GOOG"),
    ("S5-120M",   "PATH/null_baseline/S5-120M/GOOG"),
    ("S5-4K",     "PATH/null_baseline/S5-4K/GOOG"),
])

# -- Experiment grid --
GRID_PAIRS = [
    (3, 5), (5, 5), (9, 5),
    (2, 10), (3, 10), (4, 10),
    (2, 15), (3, 15),
    (1, 20), (2, 20),
]
VOLUMES = [75, 300, 485]

In [ ]:
# ── Data I/O helpers ───────────────────────────────────────────────────

def discover_v2_folders(buy_path, sell_path):
    """Discover experiment folders from directory structure."""
    folders = []
    buy_root = Path(buy_path)
    sell_root = Path(sell_path)
    if not buy_root.exists():
        print(f"  WARNING: {buy_root} not found")
        return pd.DataFrame()
    for bp in sorted(buy_root.iterdir()):
        if not bp.is_dir(): continue
        name = bp.name
        sp = sell_root / name
        if not sp.exists(): continue
        m = re.match(r'i(\d+)_c(\d+)_mb(\d+)_v(\d+)_cntxt(\d+)%', name)
        if not m: continue
        folders.append(dict(
            folder=name, i=int(m.group(1)), c=int(m.group(2)),
            mb=int(m.group(3)), vol=int(m.group(4)), cntxt_pct=int(m.group(5)),
            buy_path=str(bp), sell_path=str(sp)))
    return pd.DataFrame(folders)

def load_folder_data(folder_row, max_samples=MAX_SAMPLES):
    """Load orderbook and message data from a folder pair."""
    buy_gen = Path(folder_row['buy_path']) / 'data_gen'
    sell_gen = Path(folder_row['sell_path']) / 'data_gen'
    buy_books, sell_books, buy_msgs, sell_msgs = [], [], [], []
    for gen_dir, books, msgs in [(buy_gen, buy_books, buy_msgs), (sell_gen, sell_books, sell_msgs)]:
        if not gen_dir.exists(): continue
        for f in sorted(gen_dir.glob('*_orderbook_*_gen_id_0.csv'))[:max_samples]:
            try: books.append(pd.read_csv(f, header=None).values)
            except: pass
        for f in sorted(gen_dir.glob('*_message_*_gen_id_0.csv'))[:max_samples]:
            try: msgs.append(pd.read_csv(f, header=None).values)
            except: pass
    return dict(buy_books=buy_books, sell_books=sell_books,
                buy_msgs=buy_msgs, sell_msgs=sell_msgs,
                n_buy=len(buy_books), n_sell=len(sell_books))

def load_all_v2(paths_dict, label=""):
    """Load all data for a model."""
    results = OrderedDict()
    for model_name, paths in paths_dict.items():
        print(f"Loading {label}{model_name}...")
        grid_df = discover_v2_folders(paths['buy'], paths['sell'])
        if grid_df.empty:
            print(f"  No data for {model_name}")
            continue
        data = {}
        for _, row in grid_df.iterrows():
            data[row['folder']] = load_folder_data(row)
        results[model_name] = dict(grid=grid_df, data=data)
        print(f"  Loaded {len(grid_df)} folders, {sum(d['n_buy'] for d in data.values())} buy samples")
    return results

In [ ]:
# ── Beta (square-root law) helpers ──────────────────────────────────────

def get_midprice_from_book(book_arr):
    """Extract mid-price: (ask_price + bid_price) / 2."""
    ask = book_arr[:, 0].astype(float)
    bid = book_arr[:, 2].astype(float)
    return (ask + bid) / 2.0

def compute_combined_impact(buy_books, sell_books, tick_size=TICK_SIZE):
    """Compute antisymmetrised impact: (buy_return - sell_return) / 2."""
    buy_returns, sell_returns = [], []
    for b in buy_books:
        mid = get_midprice_from_book(b)
        mid = mid[mid > 0]
        if len(mid) < 2: continue
        buy_returns.append((mid - mid[0]) / tick_size)
    for b in sell_books:
        mid = get_midprice_from_book(b)
        mid = mid[mid > 0]
        if len(mid) < 2: continue
        sell_returns.append((mid - mid[0]) / tick_size)
    return buy_returns, sell_returns

def extract_point_cloud(data, grid_df):
    """Extract (Q, I, sigma) point cloud for beta regression."""
    points = []
    for _, row in grid_df.iterrows():
        folder = row['folder']
        if folder not in data: continue
        fd = data[folder]
        buy_rets, sell_rets = compute_combined_impact(fd['buy_books'], fd['sell_books'])
        n_pairs = min(len(buy_rets), len(sell_rets))
        for j in range(n_pairs):
            br, sr = buy_rets[j], sell_rets[j]
            L = min(len(br), len(sr))
            if L < 2: continue
            insert_end = min(row['i'] * (row['mb'] + 1), L - 1)
            combined = (br[insert_end] - sr[insert_end]) / 2.0
            if abs(combined) < 1e-10 or combined < 0: continue
            points.append(dict(Q=row['i'] * row['vol'], I=combined,
                              vol=row['vol'], i=row['i'], mb=row['mb'], sample_id=j))
    return pd.DataFrame(points)

def compute_global_beta(pc_df, daily_vol=1e6, daily_sigma=1.0):
    """Fit through-origin OLS in log-log space."""
    if pc_df.empty: return dict(beta=np.nan, r2=np.nan, n=0, ci_lo=np.nan, ci_hi=np.nan)
    x = np.log(pc_df['Q'].values / daily_vol)
    y = np.log(pc_df['I'].values / daily_sigma)
    beta = np.dot(x, y) / np.dot(x, x)
    y_pred = beta * x
    r2 = 1 - np.sum((y - y_pred)**2) / np.sum(y**2)
    return dict(beta=beta, r2=r2, n=len(pc_df), ci_lo=np.nan, ci_hi=np.nan)

def bootstrap_beta(pc_df, n_boot=N_BOOTSTRAP, daily_vol=1e6, daily_sigma=1.0):
    """Bootstrap beta by resampling sample_ids."""
    if pc_df.empty: return np.array([])
    sample_ids = pc_df['sample_id'].unique()
    rng = np.random.default_rng(42)
    betas = []
    for _ in range(n_boot):
        boot_ids = rng.choice(sample_ids, size=len(sample_ids), replace=True)
        boot_df = pd.concat([pc_df[pc_df['sample_id'] == sid] for sid in boot_ids], ignore_index=True)
        betas.append(compute_global_beta(boot_df, daily_vol, daily_sigma)['beta'])
    return np.array(betas)

In [ ]:
# ── Master curves, relaxation ─────────────────────────────────────────

def compute_master_curve(buy_data, sell_data, folder, aggr_gen,
                          tick_size=TICK_SIZE, n_vol_u=200):
    """Compute sigma-normalised volume-time master curve."""
    i_val = int(re.search(r'i(\d+)', folder).group(1))
    mb_val = int(re.search(r'mb(\d+)', folder).group(1))
    L = i_val * (mb_val + 1)
    buy_rets, sell_rets = compute_combined_impact(
        buy_data['buy_books'], sell_data['sell_books'] if sell_data else buy_data['sell_books'], tick_size)
    n_pairs = min(len(buy_rets), len(sell_rets))
    if n_pairs == 0: return None
    max_len = max(max(len(r) for r in buy_rets), max(len(r) for r in sell_rets))
    u_grid = np.linspace(0, max_len / L, n_vol_u)
    combined_curves = []
    for j in range(n_pairs):
        br, sr = buy_rets[j], sell_rets[j]
        min_len = min(len(br), len(sr))
        combined = (br[:min_len] - sr[:min_len]) / 2.0
        u_raw = np.arange(min_len) / L
        combined_curves.append(np.interp(u_grid, u_raw, combined))
    curves = np.array(combined_curves)
    return dict(u=u_grid, mean=np.nanmean(curves, axis=0),
                std=np.nanstd(curves, axis=0), n=n_pairs, folder=folder)

def compute_relaxation_ratio(master_curve, u_peak=1.0, u_final=3.0):
    """Compute I_final / I_peak."""
    if master_curve is None: return np.nan
    u, m = master_curve['u'], master_curve['mean']
    peak_idx = np.argmin(np.abs(u - u_peak))
    final_idx = np.argmin(np.abs(u - u_final))
    if m[peak_idx] < 1e-10: return np.nan
    return m[final_idx] / m[peak_idx]

In [ ]:
# ── Stability (3-method vote) ─────────────────────────────────────────
TAIL_FRAC = 0.20
SLOPE_THRESH = 0.05
WINDOW_THRESH = 0.03
CONVERGE_THRESH = 0.95

def stability_for_folder(master_curve, u_peak=1.0):
    """3-method stability vote on post-peak master curve."""
    if master_curve is None: return dict(stable=False, votes=0, methods=[False, False, False])
    u, m = master_curve['u'], master_curve['mean']
    peak_idx = np.argmin(np.abs(u - u_peak))
    post = m[peak_idx:]
    if len(post) < 10: return dict(stable=False, votes=0, methods=[False, False, False])
    n = len(post)
    tail = post[int(n * (1 - TAIL_FRAC)):]
    # Method 1: trailing slope
    x_tail = np.arange(len(tail))
    slope = np.polyfit(x_tail, tail, 1)[0] if len(tail) > 1 else 1.0
    m1 = abs(slope) / (abs(np.mean(tail)) + 1e-10) < SLOPE_THRESH
    # Method 2: two-window comparison
    mid = n // 2
    w1 = np.mean(post[max(0,mid-n//8):mid+n//8])
    w2 = np.mean(tail)
    m2 = abs(w1 - w2) / (abs(w1) + 1e-10) < WINDOW_THRESH
    # Method 3: exponential fit
    try:
        def exp_decay(x, a, b, c): return a * np.exp(-b * x) + c
        popt, _ = optimize.curve_fit(exp_decay, np.arange(n), post,
                                      p0=[post[0]-post[-1], 0.1, post[-1]], maxfev=5000)
        m3 = (1 - abs(popt[0]*np.exp(-popt[1]*n))/(abs(popt[2])+1e-10)) > CONVERGE_THRESH
    except: m3 = False
    votes = sum([m1, m2, m3])
    return dict(stable=votes >= 2, votes=votes, methods=[m1, m2, m3])

In [ ]:
# ── Decay function fitting ────────────────────────────────────────────

def fit_decay(u_grid, mean_curve, u_peak=1.0):
    """Fit power-law and exponential decay to post-peak curve."""
    peak_idx = np.argmin(np.abs(u_grid - u_peak))
    post_u = u_grid[peak_idx:] - u_peak
    post_y = mean_curve[peak_idx:]
    if len(post_y) < 5 or post_y[0] < 1e-10:
        return dict(gamma_power=np.nan, gamma_exp=np.nan)
    y_norm = post_y / post_y[0]
    try:
        def power_law(u, gamma, c): return c * (1 + u)**(-gamma)
        mask = post_u > 0
        popt, _ = optimize.curve_fit(power_law, post_u[mask], y_norm[mask], p0=[0.5, 1.0], maxfev=5000)
        gamma_power = popt[0]
    except: gamma_power = np.nan
    try:
        def exp_decay(u, a, b, c): return a * np.exp(-b * u) + c
        popt, _ = optimize.curve_fit(exp_decay, post_u, y_norm, p0=[0.3, 0.5, 0.7], maxfev=5000)
        gamma_exp = popt[1]
    except: gamma_exp = np.nan
    return dict(gamma_power=gamma_power, gamma_exp=gamma_exp)

In [ ]:
# ── NEW: Hurst Exponent of Order Flow ───────────────────────────────

def extract_order_signs(msgs_list):
    """Extract order signs: buy MO = +1, sell MO = -1."""
    all_signs = []
    for msgs in msgs_list:
        mask_mo = msgs[:, 1] == 4
        if mask_mo.sum() < 2: continue
        signs = np.where(msgs[mask_mo, 2] == 0, 1, -1)
        all_signs.append(signs)
    return all_signs

def compute_hurst_dfa(signs, max_lag=200):
    """Detrended Fluctuation Analysis for Hurst exponent."""
    if len(signs) < max_lag * 2: return np.nan
    cumsum = np.cumsum(signs - np.mean(signs))
    scales = np.unique(np.logspace(1, np.log10(max_lag), 20).astype(int))
    scales = scales[scales >= 4]
    flucts = []
    for scale in scales:
        n_seg = len(cumsum) // scale
        if n_seg < 1: continue
        F2 = 0
        for seg in range(n_seg):
            segment = cumsum[seg*scale:(seg+1)*scale]
            x = np.arange(scale)
            trend = np.polyval(np.polyfit(x, segment, 1), x)
            F2 += np.mean((segment - trend)**2)
        flucts.append(np.sqrt(F2 / n_seg))
    if len(flucts) < 3: return np.nan
    log_s = np.log(scales[:len(flucts)])
    log_f = np.log(flucts)
    H, _ = np.polyfit(log_s, log_f, 1)
    return H

def compute_autocorrelation(signs, max_lag=200):
    """Compute autocorrelation C(l) of order signs."""
    signs = np.array(signs, dtype=float)
    signs = signs - signs.mean()
    n = len(signs)
    var = np.var(signs)
    if var < 1e-10: return np.zeros(max_lag)
    acf = np.zeros(max_lag)
    for lag in range(min(max_lag, n - 1)):
        acf[lag] = np.mean(signs[:n-lag] * signs[lag:]) / var
    return acf

def compute_hurst_from_acf(signs, max_lag=200):
    """Estimate H from ACF power-law fit: C(l) ~ l^(2H-2)."""
    acf = compute_autocorrelation(signs, max_lag)
    lags = np.arange(2, max_lag)
    acf_pos = acf[2:max_lag]
    mask = acf_pos > 0
    if mask.sum() < 5: return np.nan, acf
    try:
        slope, _ = np.polyfit(np.log(lags[mask]), np.log(acf_pos[mask]), 1)
        H = (slope + 2) / 2
    except: H = np.nan
    return H, acf

In [ ]:
# ── NEW: Propagator Function G(l) ──────────────────────────────────

def compute_propagator(msgs_list, books_list, tick_size=TICK_SIZE, max_lag=200):
    """Compute G(l) = E[dp(t+l) * eps(t)]."""
    G_sum = np.zeros(max_lag)
    G_count = np.zeros(max_lag)
    for msgs, books in zip(msgs_list, books_list):
        if len(msgs) < max_lag + 10: continue
        mid = get_midprice_from_book(books) / tick_size
        dp = np.diff(mid)
        eps = np.zeros(len(msgs))
        eps[(msgs[:, 1] == 4) & (msgs[:, 2] == 0)] = 1
        eps[(msgs[:, 1] == 4) & (msgs[:, 2] == 1)] = -1
        n = min(len(dp), len(eps) - 1)
        for lag in range(min(max_lag, n)):
            valid = n - lag
            G_sum[lag] += np.sum(dp[lag:lag+valid] * eps[:valid])
            G_count[lag] += valid
    return np.where(G_count > 0, G_sum / G_count, 0), np.arange(max_lag)

In [ ]:
# ── NEW: Spread Dynamics During Impact ────────────────────────────

def compute_spread_trajectory(buy_books, sell_books, n_injection_msgs, n_total_msgs):
    """Extract bid-ask spread trajectory during impact."""
    all_spreads = []
    for books in buy_books + sell_books:
        if len(books) < 10: continue
        spread = (books[:, 0].astype(float) - books[:, 2].astype(float)) / TICK_SIZE
        spread[spread <= 0] = np.nan
        spread[spread > 100] = np.nan
        all_spreads.append(spread)
    if not all_spreads: return None
    max_len = max(len(s) for s in all_spreads)
    L = max(n_injection_msgs, 1)
    u_grid = np.linspace(0, max_len / L, 200)
    interp_spreads = []
    for s in all_spreads:
        u_raw = np.arange(len(s)) / L
        valid = ~np.isnan(s)
        if valid.sum() < 5: continue
        interp_spreads.append(np.interp(u_grid, u_raw[valid], s[valid]))
    if not interp_spreads: return None
    arr = np.array(interp_spreads)
    return dict(u=u_grid, mean=np.nanmean(arr, axis=0), std=np.nanstd(arr, axis=0), n=len(interp_spreads))

In [ ]:
# ── Load all data ─────────────────────────────────────────────────
print("="*60)
print("Loading GOOG Jan 2026 data...")
print("="*60)
R_GOOG = load_all_v2(GOOG_PATHS, label="GOOG/")

print("\n" + "="*60)
print("Loading INTC Jan 2026 data...")
print("="*60)
R_INTC = load_all_v2(INTC_PATHS, label="INTC/")

R = R_GOOG  # primary

---
## Part A: Core Metrics — Main Text

### Null Baseline

In [ ]:
# ── Null Baseline Results ───────────────────────────────────────────
print("="*60)
print("NULL BASELINE: No-injection control")
print("="*60)
null_results = OrderedDict()
for model_name, null_path in NULL_PATHS.items():
    p = Path(null_path)
    if not p.exists():
        print(f"  {model_name}: path not found ({null_path})")
        continue
    gen_dir = p / 'data_gen'
    if not gen_dir.exists(): continue
    ob_files = sorted(gen_dir.glob('*_orderbook_*_gen_id_0.csv'))[:MAX_SAMPLES]
    drifts = []
    for f in ob_files:
        try:
            df = pd.read_csv(f, header=None)
            mid = get_midprice_from_book(df.values)
            mid = mid[mid > 0]
            if len(mid) < 10: continue
            drifts.append((mid[-1] - mid[0]) / TICK_SIZE)
        except: pass
    if drifts:
        drifts = np.array(drifts)
        null_results[model_name] = dict(
            mean_drift=np.mean(drifts), std_drift=np.std(drifts),
            mean_abs_drift=np.mean(np.abs(drifts)),
            median_abs_drift=np.median(np.abs(drifts)), n=len(drifts))
        print(f"  {model_name}: drift = {np.mean(drifts):.3f} ± {np.std(drifts):.3f}, "
              f"|drift| = {np.mean(np.abs(drifts)):.3f} (n={len(drifts)})")
if not null_results:
    print("  No null baseline data found. Run experiments first.")

In [ ]:
# ── Null Baseline Figure ────────────────────────────────────────────
if null_results:
    fig = go.Figure()
    models = list(null_results.keys())
    fig.add_trace(go.Bar(
        x=models,
        y=[null_results[m]['mean_abs_drift'] for m in models],
        error_y=dict(type='data', array=[null_results[m]['std_drift'] for m in models]),
        marker_color=[MODEL_META.get(m, {}).get('color', '#888') for m in models],
        name='|Drift| (no injection)'))
    fig.update_layout(title="Null Baseline: Mean Absolute Drift Without Injection",
                      yaxis_title="Absolute drift (ticks)",
                      template=TEMPLATE, font=FONT, width=SINGLE_W, height=FIG_H)
    fig.show()
    save_fig(fig, "null_baseline_drift", w=SINGLE_W)

In [ ]:
# ── Table 1: Global Beta Comparison ───────────────────────────────
rows = []
for label, rd in R.items():
    pc = extract_point_cloud(rd['data'], rd['grid'])
    res = compute_global_beta(pc)
    boots = bootstrap_beta(pc)
    ci = (np.percentile(boots, 2.5), np.percentile(boots, 97.5)) if len(boots) > 0 else (np.nan, np.nan)
    rows.append(dict(Model=label, beta=res['beta'], R2=res['r2'], N=res['n'], CI_lo=ci[0], CI_hi=ci[1]))
beta_df = pd.DataFrame(rows).sort_values('beta')
print("\n-- Table 1: Global Beta (through-origin OLS) --")
print(f"{'Model':<20} {'β':>8} {'R²':>8} {'N':>10} {'95% CI':>20}")
for _, r in beta_df.iterrows():
    ci = f"[{r['CI_lo']:.3f}, {r['CI_hi']:.3f}]" if not np.isnan(r['CI_lo']) else "---"
    print(f"{r['Model']:<20} {r['beta']:>8.3f} {r['R2']:>8.3f} {r['N']:>10,.0f} {ci:>20}")

In [ ]:
# ── Figure: Beta Regression Lines ─────────────────────────────────
fig = go.Figure()
for label, rd in R.items():
    pc = extract_point_cloud(rd['data'], rd['grid'])
    if pc.empty: continue
    res = compute_global_beta(pc)
    meta = MODEL_META.get(label, {})
    x = np.log(pc['Q'].values / 1e6)
    y = np.log(pc['I'].values / 1.0)
    fig.add_trace(go.Scatter(x=x, y=y, mode='markers', name=label,
        marker=dict(size=3, color=meta.get('color','#888'), opacity=0.3), showlegend=False))
    x_line = np.array([x.min(), x.max()])
    fig.add_trace(go.Scatter(x=x_line, y=res['beta']*x_line, mode='lines',
        name=f"{label} (β={res['beta']:.3f})",
        line=dict(color=meta.get('color','#888'), dash=meta.get('dash','solid'), width=2)))
x_th = np.array([-15, -5])
fig.add_trace(go.Scatter(x=x_th, y=0.5*x_th, mode='lines', name='Theory β=0.5',
    line=dict(color='black', dash='dash', width=1)))
fig.update_layout(title="Beta Regression (log-log)", template=TEMPLATE, font=FONT,
    xaxis_title="ln(Q/V)", yaxis_title="ln(I/σ)", width=FULL_W, height=FIG_H)
fig.show()
save_fig(fig, "beta_regression_7m")

In [ ]:
# ── Bootstrap Beta Distributions ──────────────────────────────────
fig = go.Figure()
for label, rd in R.items():
    pc = extract_point_cloud(rd['data'], rd['grid'])
    boots = bootstrap_beta(pc)
    if len(boots) == 0: continue
    meta = MODEL_META.get(label, {})
    fig.add_trace(go.Violin(y=boots, name=label, box_visible=True,
        line_color=meta.get('color','#888'), meanline_visible=True))
fig.add_hline(y=0.5, line_dash="dash", line_color="black", annotation_text="β=0.5")
fig.update_layout(title="Bootstrap Beta (1000 resamples)",
    template=TEMPLATE, font=FONT, width=FULL_W, height=FIG_H, yaxis_title="β")
fig.show()
save_fig(fig, "bootstrap_beta_7m")

In [ ]:
# ── Master Curves per model (panel grid) ─────────────────────────
n_models = len(R)
n_cols = min(4, n_models)
n_rows = math.ceil(n_models / n_cols)
fig = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=list(R.keys()),
                    shared_yaxes=True, horizontal_spacing=0.04)
for idx, (label, rd) in enumerate(R.items()):
    row, col = idx // n_cols + 1, idx % n_cols + 1
    meta = MODEL_META.get(label, {})
    for _, frow in rd['grid'].iterrows():
        folder = frow['folder']
        if folder not in rd['data']: continue
        mc = compute_master_curve(rd['data'][folder], rd['data'][folder], folder, None)
        if mc is None: continue
        fig.add_trace(go.Scatter(x=mc['u'], y=mc['mean'], mode='lines',
            line=dict(color=meta.get('color','#888'), width=1), opacity=0.5, showlegend=False),
            row=row, col=col)
    fig.add_vline(x=1.0, line_dash="dash", line_color="gray", row=row, col=col)
fig.update_layout(title="Master Curves per Model", template=TEMPLATE, font=FONT,
    width=FULL_W, height=FIG_H * n_rows // 2)
fig.show()
save_fig(fig, "master_curves_7m", h=FIG_H * n_rows // 2)

In [ ]:
# ── Average Master Curve (7 models overlaid) ─────────────────────
fig = go.Figure()
for label, rd in R.items():
    meta = MODEL_META.get(label, {})
    all_curves = []
    for _, frow in rd['grid'].iterrows():
        folder = frow['folder']
        if folder not in rd['data']: continue
        mc = compute_master_curve(rd['data'][folder], rd['data'][folder], folder, None)
        if mc is not None: all_curves.append(mc)
    if not all_curves: continue
    u = all_curves[0]['u']
    means = np.array([c['mean'] for c in all_curves])
    avg, std = np.nanmean(means, axis=0), np.nanstd(means, axis=0)
    fig.add_trace(go.Scatter(x=u, y=avg, mode='lines', name=label,
        line=dict(color=meta.get('color','#888'), dash=meta.get('dash','solid'), width=2)))
    fig.add_trace(go.Scatter(x=np.concatenate([u, u[::-1]]),
        y=np.concatenate([avg+std, (avg-std)[::-1]]),
        fill='toself', fillcolor=meta.get('color','#888'), opacity=0.1,
        line=dict(width=0), showlegend=False))
fig.add_vline(x=1.0, line_dash="dash", line_color="gray", annotation_text="u=1")
fig.update_layout(title="Average Master Curve (7 models)", template=TEMPLATE, font=FONT,
    xaxis_title="Volume time u = n/L", yaxis_title="I/σ", width=FULL_W, height=FIG_H)
fig.show()
save_fig(fig, "avg_master_curve_7m")

In [ ]:
# ── Relaxation Ratio ──────────────────────────────────────────────
print("\n-- Relaxation Ratio: I_final / I_peak --")
print(f"{'Model':<20} {'Median r':>10} {'Mean ± std':>20} {'CV':>8} {'|Δ| from 2/3':>14}")
for label, rd in R.items():
    ratios = []
    for _, frow in rd['grid'].iterrows():
        folder = frow['folder']
        if folder not in rd['data']: continue
        mc = compute_master_curve(rd['data'][folder], rd['data'][folder], folder, None)
        r = compute_relaxation_ratio(mc)
        if not np.isnan(r): ratios.append(r)
    if ratios:
        arr = np.array(ratios)
        print(f"{label:<20} {np.median(arr):>10.2f} {np.mean(arr):>8.2f} ± {np.std(arr):.2f}     "
              f"{np.std(arr)/(abs(np.mean(arr))+1e-10):>8.2f} {abs(np.mean(arr)-2/3):>14.2f}")

In [ ]:
# ── Relaxation Ratio Figure ───────────────────────────────────────
fig = go.Figure()
for label, rd in R.items():
    meta = MODEL_META.get(label, {})
    ratios = []
    for _, frow in rd['grid'].iterrows():
        folder = frow['folder']
        if folder not in rd['data']: continue
        mc = compute_master_curve(rd['data'][folder], rd['data'][folder], folder, None)
        r = compute_relaxation_ratio(mc)
        if not np.isnan(r): ratios.append(r)
    if ratios:
        fig.add_trace(go.Box(y=ratios, name=label, marker_color=meta.get('color','#888')))
fig.add_hline(y=2/3, line_dash="dash", line_color="black", annotation_text="2/3 (theory)")
fig.update_layout(title="Relaxation Ratio (I_final/I_peak at u=3)",
    template=TEMPLATE, font=FONT, width=FULL_W, height=FIG_H, yaxis_title="r")
fig.show()
save_fig(fig, "relaxation_ratio_7m")

In [ ]:
# ── Stability Analysis ────────────────────────────────────────────
print("\n-- Stability (3-method vote) --")
print(f"{'Model':<20} {'Stable':>8} {'Total':>8} {'Fraction':>10}")
for label, rd in R.items():
    stable_count = total_count = 0
    for _, frow in rd['grid'].iterrows():
        folder = frow['folder']
        if folder not in rd['data']: continue
        mc = compute_master_curve(rd['data'][folder], rd['data'][folder], folder, None)
        s = stability_for_folder(mc)
        total_count += 1
        if s['stable']: stable_count += 1
    print(f"{label:<20} {stable_count:>8} {total_count:>8} {stable_count/max(total_count,1):>10.1%}")

In [ ]:
# ── No-Arbitrage Scorecard ────────────────────────────────────────
print("\n-- No-Arbitrage Scorecard (5 tests) --")
print(f"{'Model':<15} {'β_perm':>8} {'r':>8} {'A':>4} {'B':>4} {'C':>4} {'D':>4} {'E':>4} {'Score':>8}")
for label, rd in R.items():
    pc = extract_point_cloud(rd['data'], rd['grid'])
    beta = compute_global_beta(pc)['beta']
    ratios, decay_gammas = [], []
    for _, frow in rd['grid'].iterrows():
        folder = frow['folder']
        if folder not in rd['data']: continue
        mc = compute_master_curve(rd['data'][folder], rd['data'][folder], folder, None)
        r = compute_relaxation_ratio(mc)
        if not np.isnan(r): ratios.append(r)
        if mc is not None:
            fd = fit_decay(mc['u'], mc['mean'])
            if not np.isnan(fd['gamma_power']): decay_gammas.append(fd['gamma_power'])
    r_mean = np.mean(ratios) if ratios else np.nan
    beta_perm = beta * (r_mean if not np.isnan(r_mean) else 1.0)
    gamma = np.mean(decay_gammas) if decay_gammas else np.nan
    A = beta < 1
    B = 0.7 <= beta_perm <= 1.3 if not np.isnan(beta_perm) else False
    C = 0.3 <= gamma <= 1.0 if not np.isnan(gamma) else False
    D = 0.5 <= r_mean <= 1.0 if not np.isnan(r_mean) else False
    E = beta <= 1/(1+2*gamma) if not np.isnan(gamma) else False
    score = sum([A,B,C,D,E])
    print(f"{label:<15} {beta_perm:>8.2f} {r_mean:>8.2f} "
          f"{'\u2713' if A else '\u2717':>4} {'\u2713' if B else '\u2717':>4} "
          f"{'\u2713' if C else '\u2717':>4} {'\u2713' if D else '\u2717':>4} "
          f"{'\u2713' if E else '\u2717':>4} {score:>6}/5")

---
### New Analyses: Hurst, Propagator, Spread

In [ ]:
# ── NEW: Hurst Exponent of Order Flow ─────────────────────────────
print("\n" + "="*60)
print("HURST EXPONENT OF ORDER FLOW")
print("="*60)
hurst_results = OrderedDict()
for label, rd in R.items():
    all_signs = []
    for folder, fd in rd['data'].items():
        for msgs in fd.get('buy_msgs', []) + fd.get('sell_msgs', []):
            all_signs.extend(extract_order_signs([msgs]))
    if not all_signs: continue
    all_concat = np.concatenate(all_signs)
    if len(all_concat) < 100: continue
    H_dfa = compute_hurst_dfa(all_concat)
    H_acf, acf = compute_hurst_from_acf(all_concat)
    hurst_results[label] = dict(H_dfa=H_dfa, H_acf=H_acf, n_signs=len(all_concat))
    print(f"  {label}: H_DFA={H_dfa:.3f}, H_ACF={H_acf:.3f} (n={len(all_concat):,})")
if hurst_results:
    print(f"\n  Empirical benchmark: H ≈ 0.7 (Lillo & Farmer 2004)")

In [ ]:
# ── Hurst Exponent Figure ──────────────────────────────────────────
if hurst_results:
    fig = go.Figure()
    models = list(hurst_results.keys())
    fig.add_trace(go.Bar(x=models,
        y=[hurst_results[m]['H_dfa'] for m in models],
        marker_color=[MODEL_META.get(m,{}).get('color','#888') for m in models], name='H (DFA)'))
    fig.add_hline(y=0.7, line_dash="dash", line_color="black", annotation_text="Empirical H≈0.7")
    fig.add_hline(y=0.5, line_dash="dot", line_color="gray", annotation_text="Random walk H=0.5")
    fig.update_layout(title="Hurst Exponent of Generated Order Flow",
        yaxis_title="H (DFA)", template=TEMPLATE, font=FONT, width=SINGLE_W, height=FIG_H)
    fig.show()
    save_fig(fig, "hurst_exponent_7m", w=SINGLE_W)

In [ ]:
# ── NEW: Propagator Function G(l) ─────────────────────────────────
print("\n" + "="*60)
print("PROPAGATOR FUNCTION G(l)")
print("="*60)
propagator_results = OrderedDict()
for label, rd in R.items():
    msgs_all, books_all = [], []
    for folder, fd in rd['data'].items():
        for m, b in zip(fd.get('buy_msgs',[]), fd.get('buy_books',[])):
            msgs_all.append(m); books_all.append(b)
        for m, b in zip(fd.get('sell_msgs',[]), fd.get('sell_books',[])):
            msgs_all.append(m); books_all.append(b)
    if not msgs_all: continue
    G, lags = compute_propagator(msgs_all, books_all, max_lag=200)
    propagator_results[label] = dict(G=G, lags=lags)
    print(f"  {label}: G(1)={G[1]:.4f}, G(10)={G[10]:.4f}, G(100)={G[100]:.4f}")

In [ ]:
# ── Propagator G(l) log-log ───────────────────────────────────────
if propagator_results:
    fig = go.Figure()
    for label, pr in propagator_results.items():
        meta = MODEL_META.get(label, {})
        G, lags = pr['G'], pr['lags']
        mask = (lags > 0) & (G > 0)
        if mask.sum() < 3: continue
        fig.add_trace(go.Scatter(x=np.log10(lags[mask]), y=np.log10(G[mask]),
            mode='lines', name=label,
            line=dict(color=meta.get('color','#888'), dash=meta.get('dash','solid'), width=2)))
    l_th = np.logspace(0, 2.3, 50)
    fig.add_trace(go.Scatter(x=np.log10(l_th), y=np.log10(l_th**(-0.5)*0.1),
        mode='lines', name='l^(-0.5) (theory)', line=dict(color='black', dash='dash', width=1)))
    fig.update_layout(title="Propagator G(l) = ⟨Δp(t+l)·ε(t)⟩",
        xaxis_title="log₁₀(l)", yaxis_title="log₁₀(G)",
        template=TEMPLATE, font=FONT, width=FULL_W, height=FIG_H)
    fig.show()
    save_fig(fig, "propagator_Gl_7m")

In [ ]:
# ── NEW: Spread Dynamics During Impact ────────────────────────────
print("\n" + "="*60)
print("SPREAD DYNAMICS DURING IMPACT")
print("="*60)
fig = go.Figure()
for label, rd in R.items():
    meta = MODEL_META.get(label, {})
    all_spread_curves = []
    for _, frow in rd['grid'].iterrows():
        folder = frow['folder']
        if folder not in rd['data']: continue
        fd = rd['data'][folder]
        n_inj = frow['i'] * (frow['mb'] + 1)
        spread = compute_spread_trajectory(fd['buy_books'], fd['sell_books'],
            n_inj, (frow['i'] + frow['i']*10) * frow['mb'])
        if spread is not None: all_spread_curves.append(spread)
    if not all_spread_curves: continue
    u = all_spread_curves[0]['u']
    means = np.array([c['mean'] for c in all_spread_curves])
    avg = np.nanmean(means, axis=0)
    fig.add_trace(go.Scatter(x=u, y=avg, mode='lines', name=label,
        line=dict(color=meta.get('color','#888'), dash=meta.get('dash','solid'), width=2)))
fig.add_vline(x=1.0, line_dash="dash", line_color="gray", annotation_text="u=1")
fig.update_layout(title="Spread Dynamics During Impact",
    xaxis_title="Volume time u", yaxis_title="Spread (ticks)",
    template=TEMPLATE, font=FONT, width=FULL_W, height=FIG_H)
fig.show()
save_fig(fig, "spread_dynamics_7m")

---
## Part B: Parameter Sensitivity — Appendix

In [ ]:
# ── mb Dominates Beta ─────────────────────────────────────────────
print("\n" + "="*60)
print("PARAMETER SENSITIVITY: mb dominates β")
print("="*60)
fig = go.Figure()
for label, rd in R.items():
    meta = MODEL_META.get(label, {})
    mb_betas = {}
    for _, frow in rd['grid'].iterrows():
        folder, mb = frow['folder'], frow['mb']
        if folder not in rd['data']: continue
        pc = extract_point_cloud({folder: rd['data'][folder]}, pd.DataFrame([frow]))
        if pc.empty: continue
        res = compute_global_beta(pc)
        if not np.isnan(res['beta']): mb_betas.setdefault(mb, []).append(res['beta'])
    if mb_betas:
        mbs = sorted(mb_betas.keys())
        fig.add_trace(go.Scatter(x=mbs, y=[np.mean(mb_betas[m]) for m in mbs],
            mode='lines+markers', name=label,
            line=dict(color=meta.get('color','#888'), width=2),
            marker=dict(symbol=meta.get('marker','circle'), size=8)))
fig.add_hline(y=0.5, line_dash="dash", line_color="black")
fig.update_layout(title="β vs mb", xaxis_title="mb", yaxis_title="β",
    template=TEMPLATE, font=FONT, width=SINGLE_W, height=FIG_H)
fig.show()
save_fig(fig, "beta_vs_mb_7m", w=SINGLE_W)

In [ ]:
# ── Per-Day Beta ─────────────────────────────────────────────────
print("\n-- Per-Day Beta Stability --")
print("  [Placeholder: will be computed when date-level data is available]")
# TODO: Group by date and compute per-day beta

---
## Part C: Cross-Stock Validation (INTC) — Appendix

In [ ]:
# ── Intel Cross-Stock Validation ───────────────────────────────────
print("\n" + "="*60)
print("CROSS-STOCK VALIDATION: Intel (INTC)")
print("="*60)
if R_INTC:
    print("\n-- INTC Beta --")
    for label, rd in R_INTC.items():
        pc = extract_point_cloud(rd['data'], rd['grid'])
        res = compute_global_beta(pc)
        print(f"  {label}: β={res['beta']:.3f}, R²={res['r2']:.3f}, N={res['n']}")
    print("\n-- INTC Relaxation --")
    for label, rd in R_INTC.items():
        ratios = []
        for _, frow in rd['grid'].iterrows():
            folder = frow['folder']
            if folder not in rd['data']: continue
            mc = compute_master_curve(rd['data'][folder], rd['data'][folder], folder, None)
            r = compute_relaxation_ratio(mc)
            if not np.isnan(r): ratios.append(r)
        if ratios:
            print(f"  {label}: r={np.mean(ratios):.2f} ± {np.std(ratios):.2f}")
    print("\n-- GOOG vs INTC --")
    print(f"{'Model':<15} {'GOOG β':>10} {'INTC β':>10}")
    for label in R_INTC:
        g_beta = compute_global_beta(extract_point_cloud(R[label]['data'], R[label]['grid']))['beta'] if label in R else np.nan
        i_beta = compute_global_beta(extract_point_cloud(R_INTC[label]['data'], R_INTC[label]['grid']))['beta']
        print(f"  {label:<15} {g_beta:>10.3f} {i_beta:>10.3f}")
else:
    print("  No INTC data available yet. Run Isambard experiments first.")

---
## Part D: Save & Summary

In [ ]:
# ── Save summary statistics ───────────────────────────────────────
summary_rows = []
for label, rd in R.items():
    pc = extract_point_cloud(rd['data'], rd['grid'])
    beta_res = compute_global_beta(pc)
    ratios, stable_count, total_count = [], 0, 0
    for _, frow in rd['grid'].iterrows():
        folder = frow['folder']
        if folder not in rd['data']: continue
        mc = compute_master_curve(rd['data'][folder], rd['data'][folder], folder, None)
        r = compute_relaxation_ratio(mc)
        if not np.isnan(r): ratios.append(r)
        s = stability_for_folder(mc)
        total_count += 1
        if s['stable']: stable_count += 1
    summary_rows.append(dict(
        Model=label, beta=beta_res['beta'], R2=beta_res['r2'], N=beta_res['n'],
        relaxation_mean=np.mean(ratios) if ratios else np.nan,
        relaxation_std=np.std(ratios) if ratios else np.nan,
        stability_frac=stable_count/max(total_count,1), n_configs=total_count))
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SAVE_DIR / "summary_statistics.csv", index=False)
print(f"\nSaved summary to {SAVE_DIR / 'summary_statistics.csv'}")
print(summary_df.to_string(index=False))